# Calcualting aggregate statistic values

In [1]:
import os
import numpy as np
import pandas as pd
from scipy.stats import t

In [2]:
DATA_PATH = 'reports'
ALL_DATA = os.path.join(DATA_PATH, 'all.csv')

In [3]:
df = pd.read_csv(ALL_DATA)

## Aggregating results by condition

In [4]:
partition_by = ['param']

In [5]:
df0 = df[partition_by+['beta']].groupby(by=partition_by).agg('mean').reset_index()
df0['std'] = df[partition_by+['beta']].groupby(by=partition_by).agg('std').reset_index()['beta']
df0['k'] = df[partition_by+['beta']].groupby(by=partition_by).agg('count').reset_index()['beta']
df0['se'] = df0['std'] / np.sqrt(df0['k'])

In [6]:
df0.head(n=50)

,param,beta,std,k,se
0,nx,0.141443,0.017831,1290,0.000496
1,ny,-0.021446,0.007478,1290,0.000208
2,self,-0.086012,0.421904,1290,0.011747


In [7]:
df0.to_csv(
    os.path.join(DATA_PATH, 'complete-parameter-estimates.csv'),
    index=False,
    encoding='utf-8'
)

## T-statistics

### Difference from zero

In [8]:
null = 0
tstat = ((df0['beta'] - null) / df0['se']).values
list(zip(df0[partition_by].values, tstat, t.sf(tstat.__abs__(), df=df0['k'].values - 1)))

[(array(['nx'], dtype=object),
  np.float64(284.90428437695044),
  np.float64(0.0)),
 (array(['ny'], dtype=object),
  np.float64(-103.00866645998912),
  np.float64(0.0)),
 (array(['self'], dtype=object),
  np.float64(-7.3222057801145395),
  np.float64(2.1410192780791368e-13))]

## Visualizations!

In [9]:
import plotly.figure_factory as ff

In [10]:
hist_data = [
    df['beta'].loc[df['param'].isin([param])].values
    for param in df['param'].unique()
]

groups = df['param'].unique().tolist()

In [11]:
fig = ff.create_distplot(hist_data,groups,show_hist=False)

In [12]:
fig.update_xaxes(range=[-2,2])

In [13]:
fig.show()